# 202: Cumulative CO2 Emissions

This notebook calculates the cumulative CO2 emissions for each scenario that achieves net-zero CO2; time-series and EOC

**Definition:** Cumulative CO2 emissions are the integral of CO2 emissions until 2100.

**Inputs:**
- `100_harmonised_infilled_data.csv` - Harmonised and infilled emissions data
- `101_netzero_timings.csv` - Net-zero year metadata for each scenario

**Outputs:**
- `202_cum_em_timeseries.csv` - Scenario metadata with cumulative CO2 over time (Gt CO2)
- `202_cum_em_EOC.csv` - Scenario metadata with cumulative CO2 in 2100(Gt CO2)

In [1]:
import os
import dotenv
import pyam
import pandas as pd

from pathlib import Path

dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


In [2]:
df = pyam.IamDataFrame(
    OUTPUT_FOLDER / "100_harmonised_infilled_data.csv"
)

[INFO] 13:48:59 - pyam.core: Reading file /Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output_revision/100_harmonised_infilled_data.csv


In [3]:
df.data

,model,scenario,region,variable,unit,year,value
0,AIM/CGE 2.0,SSP1-19,World,Emissions|BC,Mt BC/yr,2023,7.219245
1,AIM/CGE 2.0,SSP1-19,World,Emissions|BC,Mt BC/yr,2024,6.908793
2,AIM/CGE 2.0,SSP1-19,World,Emissions|BC,Mt BC/yr,2025,6.602866
3,AIM/CGE 2.0,SSP1-19,World,Emissions|BC,Mt BC/yr,2026,6.301464
4,AIM/CGE 2.0,SSP1-19,World,Emissions|BC,Mt BC/yr,2027,6.004589
...,...,...,...,...,...,...,...
6378757,WITCH-GLOBIOM 4.4,CD-LINKS-No-Policy,World,Emissions|cC4F8,kt cC4F8/yr,2096,1.260864
6378758,WITCH-GLOBIOM 4.4,CD-LINKS-No-Policy,World,Emissions|cC4F8,kt cC4F8/yr,2097,1.257085
6378759,WITCH-GLOBIOM 4.4,CD-LINKS-No-Policy,World,Emissions|cC4F8,kt cC4F8/yr,2098,1.253312
6378760,WITCH-GLOBIOM 4.4,CD-LINKS-No-Policy,World,Emissions|cC4F8,kt cC4F8/yr,2099,1.249543


## Load Data

Load net-zero timings as scenario metadata.

In [4]:
df.load_meta(
    OUTPUT_FOLDER / "101_netzero_timings.csv"
)

## Filter and Aggregate CO2 Emissions

Filter for scenarios that achieve net-zero CO2.

In [5]:
df_netzero = (
    df
    .filter(achieve_netzero_co2=True)
)

Sum `Emissions|CO2|Fossil` and `Emissions|CO2|Biosphere` to get total `Emissions|CO2`.

In [6]:
df_co2 = df_netzero.add(
    a="Emissions|CO2|Fossil",
    b="Emissions|CO2|Biosphere",
    name='Emissions|CO2'
)

Convert units from Mt CO2/yr to Gt CO2/yr for cumulative calculation.

In [7]:
df_co2.convert_unit(
    current="Mt CO2/yr",
    to="Gt CO2/yr",
    inplace=True
)

In [8]:
df_co2.meta

year_netzero_co2  \
model             scenario                                    
AIM/CGE 2.0       SSP1-19                            2056.0   
                  SSP2-19                            2052.0   
                  SSP2-26                            2074.0   
                  SSP3-34                            2089.0   
                  SSP4-26                            2084.0   
...                                                     ...   
WITCH-GLOBIOM 4.4 CD-LINKS-INDC2030i_1600            2088.0   
                  CD-LINKS-NDC2030i_1000             2066.0   
                  CD-LINKS-NPi2020_1000              2076.0   
                  CD-LINKS-NPi2020_1600              2095.0   
                  CD-LINKS-NPi2020_400               2055.0   

                                           year_netzero_kyoto_ar6gwp100  \
model             scenario                                                
AIM/CGE 2.0       SSP1-19                                        2069.0   
                  SSP2-19                                        2061.0   
                  SSP2-26                                           NaN   
                  SSP3-34                                           NaN   
                  SSP4-26                                           NaN   
...                                                                 ...   
WITCH-GLOBIOM 4.4 CD-LINKS-INDC2030i_1600                           NaN   
                  CD-LINKS-NDC2030i_1000                         2079.0   
                  CD-LINKS-NPi2020_1000                          2087.0   
                  CD-LINKS-NPi2020_1600                             NaN   
                  CD-LINKS-NPi2020_400                           2069.0   

                                           achieve_netzero_co2  \
model             scenario                                       
AIM/CGE 2.0       SSP1-19                                 True   
                  SSP2-19                                 True   
                  SSP2-26                                 True   
                  SSP3-34                                 True   
                  SSP4-26                                 True   
...                                                        ...   
WITCH-GLOBIOM 4.4 CD-LINKS-INDC2030i_1600                 True   
                  CD-LINKS-NDC2030i_1000                  True   
                  CD-LINKS-NPi2020_1000                   True   
                  CD-LINKS-NPi2020_1600                   True   
                  CD-LINKS-NPi2020_400                    True   

                                           achieve_netzero_ghg  
model             scenario                                      
AIM/CGE 2.0       SSP1-19                                 True  
                  SSP2-19                                 True  
                  SSP2-26                                False  
                  SSP3-34                                False  
                  SSP4-26                                False  
...                                                        ...  
WITCH-GLOBIOM 4.4 CD-LINKS-INDC2030i_1600                False  
                  CD-LINKS-NDC2030i_1000                  True  
                  CD-LINKS-NPi2020_1000                   True  
                  CD-LINKS-NPi2020_1600                  False  
                  CD-LINKS-NPi2020_400                    True  

[864 rows x 4 columns]

## Calculate Cumulative Net-Negative Emissions

Use `pyam.timeseries.cumulative()` to integrate CO2 emissions from each scenario's net-zero year to 2100. This function performs linear interpolation between data points.

In [9]:
cumulative_co2 = df_co2.timeseries().apply(
    lambda x: pyam.timeseries.cumulative(
        x,
        first_year=2023,
        last_year=df_co2.meta.loc[x.name[0:2], "year_netzero_co2"]
    ).round(2),
    axis=1
)

In [10]:
cumulative_netnegative_co2 = df_co2.timeseries().apply(
    lambda x: pyam.timeseries.cumulative(
        x,
        first_year=df_co2.meta.loc[x.name[0:2], "year_netzero_co2"],
        last_year=2100
    ).round(2),
    axis=1
)

In [11]:
cumulative_co2

model              scenario                 region  variable       unit     
AIM/CGE 2.0        SSP1-19                  World   Emissions|CO2  Gt CO2/yr     515.26
                   SSP2-19                  World   Emissions|CO2  Gt CO2/yr     460.35
                   SSP2-26                  World   Emissions|CO2  Gt CO2/yr     831.55
                   SSP3-34                  World   Emissions|CO2  Gt CO2/yr    1373.02
                   SSP4-26                  World   Emissions|CO2  Gt CO2/yr     981.94
                                                                                 ...   
WITCH-GLOBIOM 4.4  CD-LINKS-INDC2030i_1600  World   Emissions|CO2  Gt CO2/yr    1203.58
                   CD-LINKS-NDC2030i_1000   World   Emissions|CO2  Gt CO2/yr     787.13
                   CD-LINKS-NPi2020_1000    World   Emissions|CO2  Gt CO2/yr     761.86
                   CD-LINKS-NPi2020_1600    World   Emissions|CO2  Gt CO2/yr    1205.97
                   CD-LINKS-NPi2020_400    

In [12]:
cumulative_netnegative_co2

model              scenario                 region  variable       unit     
AIM/CGE 2.0        SSP1-19                  World   Emissions|CO2  Gt CO2/yr   -163.45
                   SSP2-19                  World   Emissions|CO2  Gt CO2/yr   -332.44
                   SSP2-26                  World   Emissions|CO2  Gt CO2/yr    -75.86
                   SSP3-34                  World   Emissions|CO2  Gt CO2/yr    -25.41
                   SSP4-26                  World   Emissions|CO2  Gt CO2/yr    -28.49
                                                                                 ...  
WITCH-GLOBIOM 4.4  CD-LINKS-INDC2030i_1600  World   Emissions|CO2  Gt CO2/yr    -31.86
                   CD-LINKS-NDC2030i_1000   World   Emissions|CO2  Gt CO2/yr   -219.94
                   CD-LINKS-NPi2020_1000    World   Emissions|CO2  Gt CO2/yr   -117.98
                   CD-LINKS-NPi2020_1600    World   Emissions|CO2  Gt CO2/yr     -6.78
                   CD-LINKS-NPi2020_400     World   E

Add cumulative net-negative CO2 as scenario metadata.

In [13]:
df_co2.set_meta(
    cumulative_co2,
    name="cumulative_co2"
)

df_co2.set_meta(
    cumulative_netnegative_co2,
    name="cumulative_netnegative_co2"
)

## Save Results

In [14]:
df_co2.meta.to_csv(
    OUTPUT_FOLDER / "202_cum_em_EOC.csv"
)

In [15]:
df_co2.data

,model,scenario,region,variable,unit,year,value
0,AIM/CGE 2.0,SSP1-19,World,Emissions|CO2,Gt CO2/yr,2023,42.339768
1,AIM/CGE 2.0,SSP1-19,World,Emissions|CO2,Gt CO2/yr,2024,39.856162
2,AIM/CGE 2.0,SSP1-19,World,Emissions|CO2,Gt CO2/yr,2025,37.388946
3,AIM/CGE 2.0,SSP1-19,World,Emissions|CO2,Gt CO2/yr,2026,34.938117
4,AIM/CGE 2.0,SSP1-19,World,Emissions|CO2,Gt CO2/yr,2027,32.503678
...,...,...,...,...,...,...,...
67387,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_400,World,Emissions|CO2,Gt CO2/yr,2096,-14.423003
67388,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_400,World,Emissions|CO2,Gt CO2/yr,2097,-14.372995
67389,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_400,World,Emissions|CO2,Gt CO2/yr,2098,-14.322987
67390,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_400,World,Emissions|CO2,Gt CO2/yr,2099,-14.272979
